# Bradley-Terry Mapper Comparison

Interactive evaluation of atom-mapping mappers using an Elo/Bradley-Terry rating system.

**Workflow:**
1. Load mapped reactions from CSV (one row per reaction, one column per mapper)
2. Filter to reactions where mappers disagree (different rdchiral_plus templates)
3. Present side-by-side reaction images for human evaluation
4. Record outcomes and update Elo ratings
5. Persist results to JSON for pause/resume across sessions

In [1]:
import csv
from pathlib import Path

import ipywidgets as widgets
from elo_rating import EloRatingSystem
from IPython.display import HTML, clear_output, display
from viz_utils import (
    draw_reaction_highlighted,
    extract_template,
    get_differing_atoms,
    prepare_comparison_data,
    templates_differ,
)

## Configuration

Set the paths and mapper configuration below. Adjust `MAPPER_COLUMNS` to match your CSV columns.

In [2]:
# --- Configuration ---

# Path to the CSV with mapped reactions
CSV_PATH = Path("mapper_comparison.csv")

# Path to persist evaluation results
RESULTS_PATH = Path("elo_results.json")

# Mapper name -> CSV column name for mapped reaction SMILES
MAPPER_COLUMNS = {
    "rxnmapper_v2": "rxnmapper_v2_mapped",
    "agavechem": "agavechem_mapped",
}

# Mapper name -> CSV column name for confidence scores (optional)
CONFIDENCE_COLUMNS = {
    # "rxnmapper_v2": "rxnmapper_v2_confidence",
    # "agavechem": "agavechem_confidence",
}

# Elo parameters
INITIAL_RATING = 1500.0
K_FACTOR = 32.0

# Set to True to see mapper names, templates, and changed atoms.
# Set to False for blind evaluation.
DEV_MODE = False

## Load Data & Filter to Disagreements

In [3]:
# Load CSV rows
with open(CSV_PATH, newline="") as f:
    reader = csv.DictReader(f)
    all_rows = list(reader)

print(f"Loaded {len(all_rows)} reactions from {CSV_PATH}")

# Filter to disagreements only
comparison_data = prepare_comparison_data(
    all_rows,
    mapper_columns=MAPPER_COLUMNS,
    confidence_columns=CONFIDENCE_COLUMNS or None,
)
print(f"{len(comparison_data)} reactions with template disagreements")

Loaded 10000 reactions from mapper_comparison.csv
3962 reactions with template disagreements


## Initialise or Load Elo Rating System

In [4]:
mapper_names = list(MAPPER_COLUMNS.keys())

if RESULTS_PATH.exists():
    elo = EloRatingSystem.load(RESULTS_PATH)
    print(f"Loaded existing ratings from {RESULTS_PATH}")
    print(elo.leaderboard_str())
else:
    elo = EloRatingSystem(
        model_names=mapper_names,
        initial_rating=INITIAL_RATING,
        k_factor=K_FACTOR,
    )
    print(f"Initialised new rating system with {len(mapper_names)} mappers")
    print(elo.leaderboard_str())

Loaded existing ratings from elo_results.json
MODEL LEADERBOARD
1. rxnmapper_v2              | Rating:  1556.0
2. agavechem                 | Rating:  1088.8
Total Comparisons: 123
Rating Spread: 467.2
Outcome Distribution:
  a_correct           : 73
  both_correct        : 6
  b_correct           : 1
  both_wrong          : 43


## Interactive Evaluation

Click a button to record your assessment. The system will automatically advance to the next comparison.

- **A correct**: Mapper A's mapping is correct, B's is wrong
- **B correct**: Mapper B's mapping is correct, A's is wrong
- **Both correct**: Both mappings are valid
- **Both wrong**: Neither mapping is correct
- **Skip**: Skip this comparison without recording

In [5]:
# Build the list of (row_index, entry_a, entry_b) pairs to evaluate
# For each disagreement reaction, generate all disagreeing mapper pairs
from itertools import combinations

eval_pairs = []
for row_idx, entries in comparison_data:
    templates = [extract_template(e.mapped_rxn) for e in entries]
    for i, j in combinations(range(len(entries)), 2):
        if templates_differ(templates[i], templates[j]):
            eval_pairs.append((row_idx, entries[i], entries[j]))

print(f"Total evaluation pairs: {len(eval_pairs)}")

# Track which pairs have been evaluated
evaluated_indices: set[int] = set()

# Load previously evaluated reaction indices from existing comparisons
if elo.comparisons:
    for comp in elo.comparisons:
        if comp.reaction_index >= 0:
            evaluated_indices.add(comp.reaction_index)
    print(f"Previously evaluated: {len(evaluated_indices)} reactions")

Total evaluation pairs: 3962
Previously evaluated: 123 reactions


In [6]:
# --- Interactive GUI ---

import base64
import random

current_pair_idx = [0]  # mutable holder for current position
# Store the currently displayed (possibly swapped) entries so that
# record_outcome uses the same order the user sees.
displayed_entries: list = [None, None]  # [entry_a, entry_b] as shown on screen


# Output areas
image_output = widgets.Output()
info_output = widgets.Output()
status_output = widgets.Output()

# Buttons
btn_a = widgets.Button(description="A correct", button_style="success")
btn_b = widgets.Button(description="B correct", button_style="danger")
btn_both = widgets.Button(description="Both correct", button_style="warning")
btn_neither = widgets.Button(description="Both wrong", button_style="")
btn_skip = widgets.Button(description="Skip", button_style="info")
btn_save = widgets.Button(description="Save results", button_style="primary")

button_row = widgets.HBox([btn_a, btn_b, btn_both, btn_neither, btn_skip, btn_save])

# Image dimensions — large to fill the notebook width
IMG_WIDTH_PX = 1600
IMG_HEIGHT_PX = 500
IMG_DISPLAY_WIDTH = 1100  # CSS width for display


def find_next_unevaluated() -> int | None:
    """Find the next eval pair whose reaction has not been evaluated."""
    for i in range(current_pair_idx[0], len(eval_pairs)):
        row_idx = eval_pairs[i][0]
        if row_idx not in evaluated_indices:
            return i
    for i in range(current_pair_idx[0]):
        row_idx = eval_pairs[i][0]
        if row_idx not in evaluated_indices:
            return i
    return None


def _png_to_img_tag(png: bytes | None) -> str | None:
    """Convert PNG bytes to an HTML <img> tag, or None if render failed."""
    if png:
        b64 = base64.b64encode(png).decode()
        return f'<img src="data:image/png;base64,{b64}" width="{IMG_DISPLAY_WIDTH}"/>'
    return None


def render_current():
    """Render the current comparison pair."""
    next_idx = find_next_unevaluated()
    if next_idx is None:
        with image_output:
            clear_output(wait=True)
            display(HTML("<h3>All comparisons evaluated!</h3>"))
        with info_output:
            clear_output(wait=True)
        with status_output:
            clear_output(wait=True)
            print(elo.leaderboard_str())
        return

    current_pair_idx[0] = next_idx
    row_idx, orig_entry_a, orig_entry_b = eval_pairs[next_idx]

    # Randomize which mapper is shown on top
    if random.random() < 0.5:
        entry_a, entry_b = orig_entry_b, orig_entry_a
    else:
        entry_a, entry_b = orig_entry_a, orig_entry_b

    # Store the displayed order so record_outcome uses the same entries
    displayed_entries[0] = entry_a
    displayed_entries[1] = entry_b

    # Compute differing atoms between the two mappings
    diff = get_differing_atoms(entry_a.mapped_rxn, entry_b.mapped_rxn)
    hl_a = None
    hl_b = None
    if diff is not None and diff.has_differences:
        hl_a = diff.reactant_a | diff.product_a
        hl_b = diff.reactant_b | diff.product_b

    # Render images stacked vertically with highlighting
    with image_output:
        clear_output(wait=True)
        png_a = draw_reaction_highlighted(
            entry_a.mapped_rxn, hl_a, width=IMG_WIDTH_PX, height=IMG_HEIGHT_PX
        )
        png_b = draw_reaction_highlighted(
            entry_b.mapped_rxn, hl_b, width=IMG_WIDTH_PX, height=IMG_HEIGHT_PX
        )

        # Skip this pair if either image failed to render
        if png_a is None or png_b is None:
            evaluated_indices.add(row_idx)
            current_pair_idx[0] += 1
            render_current()
            return

        label_a = entry_a.mapper_name if DEV_MODE else "Mapper A"
        label_b = entry_b.mapper_name if DEV_MODE else "Mapper B"

        html_parts = [
            '<div style="text-align:center;">',
            f'<p><b>{label_a}</b></p>',
            _png_to_img_tag(png_a),
            '<hr style="margin:12px 0;">',
            f'<p><b>{label_b}</b></p>',
            _png_to_img_tag(png_b),
            "</div>",
        ]
        display(HTML("".join(html_parts)))

    # Render info (only in dev mode)
    with info_output:
        clear_output(wait=True)
        if DEV_MODE:
            tmpl_a = extract_template(entry_a.mapped_rxn)
            tmpl_b = extract_template(entry_b.mapped_rxn)

            print(f"Reaction index: {row_idx}")
            print(f"\n--- Mapper A: {entry_a.mapper_name} ---")
            if entry_a.confidence is not None:
                print(f"Confidence: {entry_a.confidence:.4f}")
            print(f"Template: {tmpl_a.smarts}")
            print(f"Changed atoms (map nums): {sorted(tmpl_a.atom_map_nums)}")
            if diff is not None:
                print(f"Differing atoms: {len(diff.reactant_a | diff.product_a)}")

            print(f"\n--- Mapper B: {entry_b.mapper_name} ---")
            if entry_b.confidence is not None:
                print(f"Confidence: {entry_b.confidence:.4f}")
            print(f"Template: {tmpl_b.smarts}")
            print(f"Changed atoms (map nums): {sorted(tmpl_b.atom_map_nums)}")
            if diff is not None:
                print(f"Differing atoms: {len(diff.reactant_b | diff.product_b)}")
        else:
            print(f"Reaction index: {row_idx}")
            if diff is not None and diff.has_differences:
                print(f"Atoms highlighted: {len(diff.product_a)} product, {len(diff.reactant_a)} reactant")

    # Render status
    with status_output:
        clear_output(wait=True)
        total_done = len(evaluated_indices)
        total = len(eval_pairs)
        print(f"Progress: {total_done}/{total} reactions evaluated")
        print()
        print(elo.leaderboard_str())


def record_outcome(outcome: str):
    """Record an outcome and advance.

    Uses the displayed entry order (which may be swapped from the
    original eval_pairs order) so that 'a_correct' always refers to
    the mapper shown as 'Mapper A' on screen.
    """
    entry_a = displayed_entries[0]
    entry_b = displayed_entries[1]
    if entry_a is None or entry_b is None:
        return
    next_idx = current_pair_idx[0]
    if next_idx >= len(eval_pairs):
        return
    row_idx = eval_pairs[next_idx][0]
    elo.update_ratings(
        entry_a.mapper_name, entry_b.mapper_name, outcome, reaction_index=row_idx
    )
    evaluated_indices.add(row_idx)
    elo.save(RESULTS_PATH)
    render_current()


def on_btn_a(_):
    record_outcome("a_correct")

def on_btn_b(_):
    record_outcome("b_correct")

def on_btn_both(_):
    record_outcome("both_correct")

def on_btn_neither(_):
    record_outcome("both_wrong")

def on_btn_skip(_):
    current_pair_idx[0] += 1
    render_current()

def on_btn_save(_):
    elo.save(RESULTS_PATH)
    with status_output:
        clear_output(wait=True)
        print(f"Saved to {RESULTS_PATH}")
        print(elo.leaderboard_str())


btn_a.on_click(on_btn_a)
btn_b.on_click(on_btn_b)
btn_both.on_click(on_btn_both)
btn_neither.on_click(on_btn_neither)
btn_skip.on_click(on_btn_skip)
btn_save.on_click(on_btn_save)

# Display the GUI
display(widgets.VBox([info_output, image_output, button_row, status_output]))
render_current()

## Final Results

Run the cell below to see the final leaderboard and export results.

In [7]:
print(elo.leaderboard_str())

# Save final results
elo.save(RESULTS_PATH)
print(f"\nResults saved to {RESULTS_PATH}")

# Export summary as CSV
summary = elo.get_statistics()

summary_path = Path("elo_summary.csv")
with open(summary_path, "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["rank", "mapper", "elo_rating"])
    for rank, (mapper, rating) in enumerate(summary["rankings"], 1):
        writer.writerow([rank, mapper, f"{rating:.1f}"])

print(f"Summary exported to {summary_path}")

MODEL LEADERBOARD
1. rxnmapper_v2              | Rating:  1556.0
2. agavechem                 | Rating:  1088.8
Total Comparisons: 123
Rating Spread: 467.2
Outcome Distribution:
  a_correct           : 73
  both_correct        : 6
  b_correct           : 1
  both_wrong          : 43

Results saved to elo_results.json
Summary exported to elo_summary.csv
